# İlaç KT RAG Chatbot (Colab Çalışma Alanı)
Bu notebook, oluşturduğumuz uygulamayı Colab üzerinde adım adım çalıştırmak için tasarlanmıştır.

## Adım 1: Gereksinimlerin Yüklenmesi

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-openai langchain-chroma chromadb gradio pypdf tiktoken pydantic python-dotenv unstructured

## Adım 2: Çevresel Değişkenler (API Anahtarları)
Aşağıdaki hücreyi çalıştırdığınızda sizden açıkça API anahtarlarınız istenecektir.

In [ ]:
import os
from getpass import getpass

os.environ['JINA_API_KEY'] = getpass('Lütfen Jina API Key girin: ')
os.environ['GOOGLE_API_KEY'] = getpass('Lütfen Google (Gemini) API Key girin: ')

## Adım 3: Dizin Yapısının Kurulması ve Kodların Oluşturulması

In [ ]:
!mkdir -p app pdfs chromadb
!touch app/__init__.py

In [ ]:
%%writefile app/ingest.py
import argparse
import os
import hashlib
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

CHROMA_DB_DIR = "./chroma_db"
MANIFEST_PATH = "./manifest.json"

def get_file_hash(filepath: Path) -> str:
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read()
        hasher.update(buf)
    return hasher.hexdigest()

def extract_drug_id(doc_path: Path, first_page_text: str) -> str:
    lines = first_page_text.split('\n')
    start_collecting = False
    drug_name_lines = []
    
    # Durmamızı söyleyecek bölüm başlangıçları veya kullanım yolu ifadeleri
    stop_prefixes = [
        "ağız",
        "oral",
        "deri",
        "kas",
        "damar",
        "etkin madde",
        "yardımcı madde",
        "ağızdan",
        "kas içine",
        "damar içine",
        "cilt üzerine",
        "deri altına",
        "bu kullanma talimatında",
        "kullanmadan önce"
    ]
    
    for line in lines:
        clean_line = line.strip()
        
        # 1. Aşama: "KULLANMA TALİMATI" başlığını bulana kadar bekle
        if not start_collecting:
            if "KULLANMA TALİMATI" in clean_line.upper():
                start_collecting = True
            continue
            
        # Boş satırları atla
        if not clean_line:
            continue
            
        # 2. Aşama: Başlık bulundu, artık satırları topla ta ki bir stop_prefix bulana kadar
        # Satır başındaki madde işareti (• vs.) temizleyip küçük harfe çevirelim
        lower_line = clean_line.lower().lstrip("•.-* ")
        
        if any(lower_line.startswith(prefix) for prefix in stop_prefixes):
            break
            
        drug_name_lines.append(clean_line)
        
    # Eğer başarıyla bulduysa bunları tek boşlukla birleştir (örn: "A-FERİN FORTE 500 mg...") ve ® sembolünü temizle
    if drug_name_lines:
        return " ".join(drug_name_lines).replace("®", "").strip()
    
    # Fallback (Bulunamazsa eski yönteme dön)
    return doc_path.stem.upper().replace("®", "").strip()

def split_kt_by_sections(text: str, drug_id: str, file_hash: str) -> list[Document]:
    # Başlıkları yakalayacak esnek regex desenleri
    patterns = {
        "1. İlaç nedir ve ne için kullanılır?": r"(?m)^\s*1\.\s+.*nedir\s+ve\s+ne\s+için\s+kullanılır",
        "2. Kullanmadan önce dikkat edilmesi gerekenler": r"(?m)^\s*2\.\s+.*kullanmadan\s+önce\s+dikkat\s+edilmesi\s+gerekenler",
        "3. Nasıl kullanılır?": r"(?m)^\s*3\.\s+.*nasıl\s+kullanılır",
        "4. Olası yan etkiler nelerdir?": r"(?m)^\s*4\.\s+.*olası\s+yan\s+etkiler",
        "5. Saklama koşulları": r"(?m)^\s*5\.\s+.*saklanması"
    }

    import re
    matches = []
    for section_name, pattern in patterns.items():
        # İlk eşleşmeyi bul
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            matches.append({"name": section_name, "start": match.start()})
            
    # Başlangıç indeksine göre sırala
    matches.sort(key=lambda x: x["start"])
    
    sections = []
    if not matches:
        # Hiç başlık bulunamazsa tüm metni tek bir genel bölüm olarak al
        sections.append({"name": "Genel Bilgiler", "content": text.strip()})
    else:
        # Bulunan bölümleri ayır
        for i in range(len(matches)):
            # İlk başlıktan önceki metni (prelude) giriş bölümü yapmak yerine ilk bölümün başına dahil ediyoruz
            start_index = 0 if i == 0 else matches[i]["start"]
            end_index = matches[i+1]["start"] if i + 1 < len(matches) else len(text)
            sections.append({
                "name": matches[i]["name"],
                "content": text[start_index:end_index].strip()
            })
            
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=150,
        separators=["\n\n", r"(?<=[.!?])\s+", "\n", " ", ""],
        is_separator_regex=True
    )
    
    docs = []
    for sec in sections:
        chunks = text_splitter.split_text(sec["content"])
        for chunk in chunks:
            # RAG performansını artırmak için ilaç adını bölüm başlığının başına ekliyoruz
            chunk_text = f"[{drug_id} - {sec['name']}]\n\n{chunk}"
            docs.append(Document(
                page_content=chunk_text,
                metadata={
                    "drug_id": drug_id,
                    "section": sec["name"],
                    "file_hash": file_hash
                }
            ))
            
    return docs

def process_pdfs(pdf_dir: str, mode: str):
    pdf_dir_path = Path(pdf_dir)
    manifest = {}
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            manifest = json.load(f)

    db = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=JinaEmbeddings(jina_api_key=os.environ.get("JINA_API_KEY"), model_name="jina-embeddings-v3"))
    
    for filepath in pdf_dir_path.glob("*.pdf"):
        file_hash = get_file_hash(filepath)
        if mode == "incremental" and manifest.get(str(filepath)) == file_hash:
            print(f"Skipping {filepath.name}, no changes.")
            continue
            
        print(f"Processing {filepath.name}...")
        loader = PyPDFLoader(str(filepath))
        docs = loader.load()
        if not docs:
            continue
            
        # İlk sayfadaki gereksiz Kullanma Talimatı uyarı bloğunu temizleme
        first_page_content = docs[0].page_content
        
        import re
        try:
            # 1. Blok: Uyarı metnini kes
            # Preamble opsiyonel: (bu ilac[ıi]\s+kullanmay\s*a\s+ba[şs]lamadan\s+[öo]nce\s+)?
            # Toleranslı yaklaşımlar: kelime arası \s+, kullanmay a gibi durumlara karşı \s*
            # Bitiş: y[üu]ksek\s+veya\s+d[üu][şs][üu]k\s+doz\s+kullanmay[ıi]n[ıi]z\.?
            pattern1 = re.compile(
                r"(?:bu\s+ilac[ıi]\s+kullanmay\s*a\s+ba[şs]lamadan\s+[öo]nce\s+)?bu\s+kullanma\s+tal[iı]mat[ıi]n[ıi].*?y[üu]ksek\s+veya\s+d[üu][şs][üu]k\s+doz\s+kullanmay[ıi]n[ıi]z\.?",
                re.IGNORECASE | re.DOTALL
            )
            first_page_content = pattern1.sub("", first_page_content)
            
            # 2. Blok: İçindekiler listesini kes (Bu Kullanma Talimatında -> Başlıkları yer almaktadır.)
            # Boşluk toleransları \s+ ile sağlandı
            pattern2 = re.compile(
                r"bu\s+kullanma\s+tal[iı]mat[ıi]nda\s*:?.*?ba[şs]l[ıi]klar[ıi]\s+yer\s+almaktad[ıi]r\.?",
                re.IGNORECASE | re.DOTALL
            )
            first_page_content = pattern2.sub("", first_page_content)
            
            docs[0].page_content = first_page_content
        except Exception as e:
            print(f"İçerik temizleme hatası ({filepath.name}): {e}")
                
        # İlaca ait kimliği güncellenmiş metinden çıkar
        drug_id = extract_drug_id(filepath, docs[0].page_content)

        # Sayfaların en başındaki tek başına duran sayfa numaralarını temizle
        for doc in docs:
            doc.page_content = re.sub(r'^\s*\d+\s*\n', '', doc.page_content)
        
        # Tüm PDF içeriğini tek bir metinde birleştir
        full_text = "\n\n".join(doc.page_content for doc in docs)
        
        try:
            # 3. Blok: Yan etkilerin raporlanması kısmını temizle (TÜFAM vb.)
            pattern_tufam = re.compile(r"Yan\s+etkilerin\s+raporlanmas[ıi].*?sa[ğg]lam[ıi][şs]\s+olacaks[ıi]n[ıi]z\.?", re.IGNORECASE | re.DOTALL)
            full_text = pattern_tufam.sub("", full_text)
            
            # 4. Blok: Ruhsat Sahibi ve Üretim Yeri (Genelde belgenin en sonudur, bu yüzden Ruhsat Sahibi'nden sonrasını tamamen silebiliriz)
            pattern_ruhsat = re.compile(r"Ruhsat\s+[Ss]ahibi\s*:.*", re.IGNORECASE | re.DOTALL)
            full_text = pattern_ruhsat.sub("", full_text)
        except Exception as e:
            print(f"İçerik temizleme hatası (full_text - {filepath.name}): {e}")
        
        # Semantik olarak başlıklarına ve parçalara ayır (sub-chunking de dahil)
        chunks = split_kt_by_sections(full_text, drug_id, file_hash)
        
        # Eski veriyi temizleme (Incremental destek için metadata filtering gerekir, 
        # Chroma'da document delete by metadata için extra logic yazılmalıdır)
        # Rate limit engellemek için parça parça veya bekleyerek yükleme yapalım
        import time
        # Önceki kod bloğundaki db.add_documents(chunks) yerine:
        batch_size = 50
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i+batch_size]
            try:
                db.add_documents(batch)
                time.sleep(2) # Jina Rate Limiti için 2 saniye bekle
            except Exception as e:
                print(f"Embedding hatası (bekleniyor...): {e}")
                time.sleep(10) # Hata alınırsa daha uzun bekle ve tekrar dene
                try:
                    db.add_documents(batch)
                except Exception as inner_e:
                    print(f"Retry başarısız oldu, atlanıyor: {inner_e}")
                    
        manifest[str(filepath)] = file_hash

    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f)
        
    print("Ingestion completed.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--pdf-dir", type=str, required=True)
    parser.add_argument("--mode", type=str, choices=["incremental", "full"], default="incremental")
    args = parser.parse_args()
    process_pdfs(args.pdf_dir, args.mode)

In [ ]:
%%writefile app/retrieval.py
import os
import re
from langchain_chroma import Chroma
from langchain_community.embeddings import JinaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

CHROMA_DB_DIR = "./chroma_db"

# Global objeler: RAG sistemi ve LLM her çağrıda yeniden oluşturulmaz (Performans Artışı)
db = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=JinaEmbeddings(jina_api_key=os.environ.get("JINA_API_KEY"), model_name="jina-embeddings-v3"))
retriever = db.as_retriever(search_kwargs={"k": 5})

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

prompt = PromptTemplate.from_template("""
Aşağıdaki sağlanan bağlamı (context) kullanarak, kullanıcının son sorusunu yanıtla.
Eğer sadece bağlamda yeterli bilgi yoksa, kesinlikle uydurma yapma ve sadece 'Bilmiyorum.' yaz. 
Kullanıcının sorusu bir ilaç adını içermiyorsa, bağlamda bulduğun ilacın adını belirterek cevapla.

Bağlam: 
{context}

Kullanıcının Sorusu: {question}
Yanıt:
""")

chain = prompt | llm | StrOutputParser()

def get_answer(query: str, history: list = None) -> tuple[str, str, str]:
    docs = retriever.invoke(query)
    
    if not docs:
         return "Bilmiyorum.", "Tespit edilemedi", ""
         
    # Dokümanlardan tespit edilen ilacı alalım
    drug_id = docs[0].metadata.get("drug_id", "Bilinmiyor")

    def format_docs(docs_list):
        return "\n\n".join(doc.page_content for doc in docs_list)

    answer = chain.invoke({
        "context": format_docs(docs),
        "question": query
    })
    
    # Gelişmiş Fail-Safe
    # "Üzgünüm, bilmiyorum.", "Maalesef bilmiyorum.", "Bilmiyorum." gibi ifadeleri tam yakalar.
    # "Bilmiyorum, ancak..." gibi devam eden cümleleri es geçer (yanlış kırpmayı önler).
    if re.fullmatch(r'(?i)^[^\w]*(üzgünüm|maalesef|hayır)?[^\w]*bilmiyorum[^\w]*$', answer.strip()):
         answer = "Bilmiyorum."
         
    # Modele gönderilen kaynak metinleri log (console) için formatlıyoruz
    used_chunks_str = ""
    for i, doc in enumerate(docs):
        section_name = doc.metadata.get('section', 'Bilinmiyor')
        used_chunks_str += f"**Parça {i+1} ({section_name}):**\n```text\n{doc.page_content}\n```\n\n"
         
    return answer, drug_id, used_chunks_str

In [ ]:
%%writefile app/ui.py
import argparse
import gradio as gr
import dotenv
from app.retrieval import get_answer

dotenv.load_dotenv()

def chat_interface(message, history):
    if not message:
        return ""
    
    # RAG sistemi 3 parametre dönüyor (Cevap, İlaç ID, Kullanılan Chunklar). history ignore ediliyor.
    answer, drug_id, chunks_str = get_answer(message, history)
    
    final_response = f"{answer}\n\n**[Tespit Edilen İlaç: {drug_id}]**"
    
    # Text chunklarını Colab hücresinin çıktısına (console) yazdırıyoruz
    if chunks_str:
        print("\n" + "="*60)
        print(f"🧐 KULLANICI SORUSU: {message}")
        print("-" * 60)
        print(f"📄 MODELE GÖNDERİLEN KAYNAK METİNLER (CHUNKLAR):\n\n{chunks_str}")
        print("="*60 + "\n")
        
    return final_response

def main(host, port, share=False):
    with gr.Blocks(title="İlaç KT Chatbot") as demo:
        gr.Markdown("## İlaç Sohbet Botu RAG Q&A")
        
        # Kullanıcıyı yönlendiren önemli uyarı mesajı eklendi
        gr.Markdown("""
        ⚠️ **ÖNEMLİ UYARI:** 
        Bu asistan geçmiş sohbetleri **hatırlamaz**. Veritabanında doğru ilacı bulabilmesi için, **Lütfen HER sorunuzda ilacın tam adını belirtin.** 
        *(Örn: 'Yan etkileri nelerdir?' yerine 'Parol'un yan etkileri nelerdir?' şeklinde sorunuz.)*
        """)
        
        chatbot = gr.ChatInterface(
            fn=chat_interface,
            chatbot=gr.Chatbot(height=400),
            textbox=gr.Textbox(placeholder="İlacın adını beilrterek sorunuzu girin... (Örn: Parol hamilelikte kullanılır mı?)", container=False, scale=7),
            title="Sadece İlaç KT PDF'lerine Dayanarak Cevap Veren Asistan",
        )
    demo.launch(server_name=host, server_port=port, share=share)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--host", type=str, default="0.0.0.0")
    parser.add_argument("--port", type=int, default=7860)
    parser.add_argument("--share", action="store_true", help="Create a public link for Gradio")
    args = parser.parse_args()
    main(args.host, args.port, True) # Colab için force share=True

## Adım 4: Gömme (Embedding) ve İndeksleme İşlemi

In [ ]:
!python -m app.ingest --pdf-dir ./pdfs --mode full

## Adım 5: Uygulamayı Başlatma
Bu sayede Gradio, Colab üzerinden public bir `.gradio.live` linki verecektir.

In [ ]:
!python -m app.ui --share